In [0]:
from pyspark.sql.functions import *

In [0]:
service_credential = dbutils.secrets.get(scope='scope-test-learning-awg', key='azure-ms-entra-secret-key-vault-governance-awg')
 
spark.conf.set("fs.azure.account.auth.type.datalaketestlearningawg.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.datalaketestlearningawg.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.datalaketestlearningawg.dfs.core.windows.net", "504d82b4-c2c8-496f-9de9-b944b4f1b9f6")
spark.conf.set("fs.azure.account.oauth2.client.secret.datalaketestlearningawg.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.datalaketestlearningawg.dfs.core.windows.net", "https://login.microsoftonline.com/50a3af05-f42e-4ad1-9203-9b4d0f7b4664/oauth2/token")

In [0]:
schemaa = """
id bigint,
case_number string,
Date timestamp,
Block string,
IUCR string,
Primary_Type string,
Description string,
Location_Description string,
Arrest string,
Domestic string,
Beat string,
District string,
Ward string,
Community_Area string,
FBI_Code string,
X_Coordinate double,
Y_Coordinate double,
Year string,
Updated_On timestamp,
Latitude string,
Longitude string,
Location string
"""

In [0]:
df = spark.read.format('csv').schema(schemaa).option('header', 'true').load('abfss://adf-swdi-s3-1@datalaketestlearningawg.dfs.core.windows.net/source/api/views/ijzp-q8t2/rows.csvf89568009b486b640412bba863bc45a11f8efc9396029c714dda494910a90164')

In [0]:
df.printSchema()

In [0]:
df.select(spark_partition_id()).distinct().count()

In [0]:
df.write.format('delta').mode('overwrite').saveAsTable('main.default.chicago_crimes2')
#option('optimizeWrite', 'true')

In [0]:
df.repartition(6, 'id')

In [0]:
df.write.format('delta').option('optimizeWrite', 'true').mode('overwrite').saveAsTable('main.default.chicago_crimes3')

In [0]:
df.write.format('delta').partitionBy('Primary_Type').option('optimizeWrite', 'true').mode('overwrite').saveAsTable('main.default.chicago_crimes4')

In [0]:
spark.conf.get("spark.sql.files.maxPartitionBytes")

In [0]:
%sql
optimize main.default.chicago_crimes2 zorder by id

In [0]:
%sql
DESCRIBE DETAIL main.default.chicago_crimes2;

In [0]:
%sql
DESCRIBE DETAIL main.default.chicago_crimes3;

In [0]:
%sql
DESCRIBE DETAIL main.default.chicago_crimes4;

In [0]:
%sql
select * from main.default.chicago_crimes1 limit 2

In [0]:
%sql
select current_user()

In [0]:
%sql
create or replace table main.default.chicago_crimes5 as select * from main.default.chicago_crimes4
cluster by id


In [0]:
%sql
OPTIMIZE main.default.chicago_crimes5 FULL

In [0]:
%sql
alter table main.default.chicago_crimes5 set tblproperties (delta.autoOptimize.optimizeWrite = true, delta.autoOptimize.autoCompact = true);

In [0]:
%sql
alter table main.default.chicago_crimes5 cluster by AUTO;

In [0]:
%sql
OPTIMIZE main.default.chicago_crimes5 FULL;

In [0]:
from pyspark.sql.functions import udtf
@udtf(returnType="splitted_block: string")
class SplitterClass:
    def eval(self, text):
        for tex in text.split():
            yield (tex,)
# udtf_text_splitter = udtf(SplitterClass, returnType="splitted_block: string")

In [0]:
spark.udtf.register("sql_udtf_text_splitter", SplitterClass)

In [0]:
%sql
select b.splitted_block,
a.* 
from main.default.chicago_crimes5 a,
lateral sql_udtf_text_splitter(a.BLOCK) as b
limit 5

In [0]:
%sql
select b.splitted_block,
a.* 
from main.default.chicago_crimes5 a
join lateral sql_udtf_text_splitter(a.BLOCK) as b
limit 5

In [0]:
%sql
select b.splitted_block,
a.* 
from main.default.chicago_crimes5 a
join lateral (select * from sql_udtf_text_splitter(a.BLOCK)) as b
limit 5

In [0]:
SplitterClass(lit("HI BYE")).show()

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf, lit
@pandas_udf("integer")
def vectorized_square(s):
    return s * s
vectorized_square(lit(5)).show()

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf, lit
@pandas_udf("integer")
def vectorized_square(s):
    return s * s
spark.createDataFrame([(5,),(6,),(7,),(8,),(9,),(5,),(5,)], ['value']).select(vectorized_square('value')).show()

In [0]:
def square_func(n):
    return n * n if n is not None else None
square_func(lit(5)).show()

In [0]:
def square_func(n):
    return n * n if n is not None else None
spark.createDataFrame([(5,),(6,),(7,),(8,),(9,),(5,),(5,)], ['value']).select(square_func('value')).show()

In [0]:
def square_func(n):
    return n * n if n is not None else None
spark.createDataFrame([(5,),(6,),(7,),(8,),(9,),(5,),(5,)], ['value']).select(square_func(col('value'))).show()

In [0]:
%sql
select D1.DeptName, jl.EmpName,jl.Salary
FROM main.default.Department D1
join lateral (
select * from main.default.Employee e1
where E1.DeptID = D1.DeptID
order by e1.salary desc
limit 2
) as jl